# OCTMNIST — Decision Consistency Evaluation (Medical Domain)

Extends DCF to medical imaging: OCTMNIST retinal OCT scans, 4 classes.
ImageNet-pretrained models evaluated on grayscale medical images — hardest cross-domain test.

Key finding: high CS + low accuracy = CAG < 0 = Consistently Wrong.
This is the most dangerous failure mode for clinical deployment.

Requires: pip install medmnist

In [ ]:
OUTPUT_DIR = r'E:\decision_consistency_octmnist_outputs'
NUM_EVAL=1000; SEED=42; ALPHA=0.5

In [ ]:
# pip install medmnist if needed
import os, json, random, io, csv, gc
import numpy as np
import torch, torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm
import medmnist
from medmnist import OCTMNIST
import warnings; warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF']='max_split_size_mb:128'
for sub in ['tables','figures/consistency','figures/gradcam']:
    os.makedirs(os.path.join(OUTPUT_DIR,sub),exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)

In [ ]:
# Auto-downloads ~55MB on first run; grayscale (1,28,28) -> 3-channel
test_dataset=OCTMNIST(split='test',transform=None,download=True)
X_test_raw,y_test_raw=[],[]
for img,label in test_dataset:
    img_np=np.array(img); X_test_raw.append(np.stack([img_np]*3,axis=0)); y_test_raw.append(int(label[0]))
X_test_raw=np.array(X_test_raw); y_test_raw=np.array(y_test_raw)
indices=np.random.permutation(len(X_test_raw))[:NUM_EVAL]
X_sub=X_test_raw[indices]; y_sub=y_test_raw[indices]
print(f'OCTMNIST subset: {X_sub.shape}  classes: {np.unique(y_sub)}')

In [ ]:
preprocess=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])
def prepare(t): return preprocess(transforms.ToPILImage()(t.cpu().clamp(0,1))).unsqueeze(0).to(device)
def apply_jpeg(t,q=75):
    buf=io.BytesIO(); transforms.ToPILImage()(t.cpu().clamp(0,1)).save(buf,format='JPEG',quality=q); buf.seek(0); return transforms.ToTensor()(Image.open(buf))
def generate_perturbations(t):
    return {'rotation':TF.rotate(t,angle=2),'brightness':TF.adjust_brightness(t,brightness_factor=1.1),'noise':torch.clamp(t+0.01*torch.randn_like(t),0,1),'contrast':TF.adjust_contrast(t,contrast_factor=1.1),'blur':TF.gaussian_blur(t,kernel_size=3,sigma=0.5),'jpeg':apply_jpeg(t)}
def infer(model,t):
    with torch.no_grad(): probs=F.softmax(model(prepare(t)),dim=1); return probs.argmax(1).item(),probs.max().item()
def compute_metrics(op,oc,pr,alpha=ALPHA):
    K=len(pr); S=sum(1 for p,_ in pr.values() if p==op)/K; D=sum(abs(c-oc)/max(oc,1-oc) for _,c in pr.values())/K
    return {'Label_Stability':S,'Confidence_Deviation':D,'Consistency_Score':alpha*S+(1-alpha)*(1-D)}
print('Pipeline ready.')

In [ ]:
model_fns={'ResNet-18':models.resnet18,'ResNet-50':models.resnet50,'VGG-16':models.vgg16,'MobileNetV2':models.mobilenet_v2}
results={'OCTMNIST':{}}
for mn,fn in model_fns.items():
    print(f'Processing {mn}...')
    model=fn(pretrained=True).eval().to(device); per_sample=[]
    for idx in tqdm(range(NUM_EVAL),desc=mn):
        img_tensor=torch.from_numpy(X_sub[idx]).float()/255.0
        op,oc=infer(model,img_tensor); pr={k:infer(model,v) for k,v in generate_perturbations(img_tensor).items()}
        m=compute_metrics(op,oc,pr); m['true_label']=int(y_sub[idx]); m['orig_pred']=op; m['orig_conf']=oc
        per_sample.append(m)
    results['OCTMNIST'][mn]=per_sample
    ls=np.mean([m['Label_Stability'] for m in per_sample]); cd=np.mean([m['Confidence_Deviation'] for m in per_sample]); cs=np.mean([m['Consistency_Score'] for m in per_sample])
    print(f'  LS={ls:.3f}  CD={cd:.3f}  CS={cs:.3f}')
    del model; torch.cuda.empty_cache(); gc.collect()
print('Done.')

In [ ]:
header=['Dataset','Model','Avg_Label_Stability','Avg_Confidence_Deviation','Avg_Consistency_Score','Std_Consistency_Score']
rows=[]; model_names=list(results['OCTMNIST'].keys())
for mn in model_names:
    ms=results['OCTMNIST'][mn]; ls=np.mean([m['Label_Stability'] for m in ms]); cd=np.mean([m['Confidence_Deviation'] for m in ms]); cs=np.mean([m['Consistency_Score'] for m in ms]); std=np.std([m['Consistency_Score'] for m in ms])
    rows.append({'Dataset':'OCTMNIST','Model':mn,'Avg_Label_Stability':round(ls,4),'Avg_Confidence_Deviation':round(cd,4),'Avg_Consistency_Score':round(cs,4),'Std_Consistency_Score':round(std,4)})
    print(f'{mn:<14} LS={ls:.3f} CD={cd:.3f} CS={cs:.3f}')
    with open(os.path.join(OUTPUT_DIR,'tables',f'raw_OCTMNIST_{mn}.json'.replace('-','_')),'w') as f: json.dump(ms,f,indent=2)
csv_path=os.path.join(OUTPUT_DIR,'tables','summary_table_octmnist.csv')
with open(csv_path,'w',newline='') as f: w=csv.DictWriter(f,fieldnames=header); w.writeheader(); w.writerows(rows)
print('Saved:',csv_path)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
# Accuracy from linear probe (notebook 12)
acc_est={'ResNet-18':0.470,'ResNet-50':0.526,'VGG-16':0.628,'MobileNetV2':0.513}
print('CAG Analysis (OCTMNIST):'); print(f'{"Model":<14} {"Acc":>7} {"CS":>7} {"CAG":>8}  Regime'); print('-'*55)
for mn in model_names:
    acc=acc_est[mn]; cs=np.mean([m['Consistency_Score'] for m in results['OCTMNIST'][mn]]); cag=acc-cs
    print(f'{mn:<14} {acc:>7.3f} {cs:>7.4f} {cag:>8.4f}  {"Overconfident" if cag>0 else "Consistently Wrong (clinical risk)"}')
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].bar(['Accuracy','CS'],[0.470,0.861],color=['tomato','steelblue']); axes[0].set_ylim(0,1); axes[0].set_title('OCTMNIST/ResNet-18: Standard vs DCF')
cag_vals=[acc_est[mn]-np.mean([m['Consistency_Score'] for m in results['OCTMNIST'][mn]]) for mn in model_names]
axes[1].bar(model_names,cag_vals,color='steelblue'); axes[1].axhline(0,color='black',lw=1.5); axes[1].set_title('All CAG<0: Consistently Wrong — Clinical Risk')
fig.suptitle('OCTMNIST: DCF Exposes Medical Deployment Risk'); plt.tight_layout()
path=os.path.join(OUTPUT_DIR,'figures','consistency','octmnist_cag.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)